# Linear L-cell Polynomial Arrangement Calculator (n=4)
### Highly Optimized Counting via Matrix Chains and GPU Acceleration

This notebook calculates the sequence of distinct 4-edged colored snakes for linear arrangements of $L$ cells up to $L=20$ and beyond. 

**Problem Formulation:**
- We consider a linear horizontal arrangement of $L$ square cells.
- Each cell has 4 edges (S, E, N, W) colored 0 or 1.
- **Validity:** Each cell must have $\sum \text{edges} \le 2$ (11 valid patterns).
- **Matching:** Consecutive cells share an edge (Hinge); edge values must match.
- **Equivalence:** Snakes are the same if related by the symmetry group $G$ consisting of:
  - Dihedral group of the square (8 transforms)
  - Global 0 $\leftrightarrow$ 1 value flip
  - Path reversal

**Methodology:**
We use **Burnside's Lemma** which states that the number of orbits is:
$$N_{orbits} = \frac{1}{|G|} \sum_{g \in G} |\text{Fix}(g)|$$

Instead of brute-force enumeration, we compute $|\text{Fix}(g)|$ using **matrix products** ($O(L)$), allowing calculation for very large $L$. We provide a GPU-accelerated implementation using PyTorch for maximum performance.

In [ ]:
import torch
import numpy as np
import time
import pandas as pd

print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Core Constants and Symmetry Group
We define the 11 valid patterns and the edge permutations for the 8 dihedral transformations.

In [ ]:
# Directions: S=0, E=1, N=2, W=3
S, E, N, W = 0, 1, 2, 3
OPP = {S: N, E: W, N: S, W: E}

# 11 patterns with sum <= 2
VALID_PATTERNS = [
    (a, b, c, d) for a in (0, 1) for b in (0, 1) for c in (0, 1) for d in (0, 1) 
    if a + b + c + d <= 2
]

# 8 Dihedral transformations (edge permutations)
EDGE_PERMS = (
    (0, 1, 2, 3), # 0 identity
    (3, 0, 1, 2), # 1 rot 90 CCW
    (2, 3, 0, 1), # 2 rot 180
    (1, 2, 3, 0), # 3 rot 270 CCW
    (2, 1, 0, 3), # 4 reflect x (y -> -y)
    (0, 3, 2, 1), # 5 reflect y (x -> -x)
    (3, 2, 1, 0), # 6 reflect y=x
    (1, 0, 3, 2), # 7 reflect y=-x
)

VALID_BOTH = [p for p in VALID_PATTERNS if sum(p) == 2]

### Linear Snake Symmetry Analysis
For a straight horizontal line, the geometric symmetries (mapping the set of cell positions to itself) are:
1. `id`: (di=0, rev=False)
2. `ref_x`: (di=4, rev=False) - swaps S/N
3. `r180`: (di=2, rev=True) - swaps E/W, S/N
4. `ref_y`: (di=5, rev=True) - swaps E/W

Each can be combined with a global `flip` ($0 \leftrightarrow 1$). The identity case is computed via transfer matrix powers. Reversal cases are computed by finding patterns that can satisfy the "folding" constraint at the center.

In [ ]:
class LinearSnakeSolver:
    def __init__(self, device):
        self.device = device
        self.patterns = torch.tensor(VALID_PATTERNS, device=device)
        self.patterns_both = torch.tensor(VALID_BOTH, device=device)

    def _get_transfer_matrix(self, pats):
        # W edge of cell i+1 matches E edge of cell i
        M = torch.zeros((2, 2), dtype=torch.double, device=self.device)
        for i in range(len(pats)):
            p = pats[i]
            M[p[W], p[E]] += 1
        return M

    def count_fixed(self, L, di, rev, flip, both_only=False):
        perm = torch.tensor(EDGE_PERMS[di], device=self.device)
        pats = self.patterns_both if both_only else self.patterns
        
        if not rev:
            # Fixed under (di, False, flip)
            fixed_pats = []
            for i in range(len(pats)):
                p = pats[i]
                tp = p[perm]
                if flip: tp = 1 - tp
                if torch.equal(p, tp): 
                    fixed_pats.append(p)
            if not fixed_pats: return 0.0
            
            fixed_pats_tensor = torch.stack(fixed_pats)
            M = torch.zeros((2, 2), dtype=torch.double, device=self.device)
            for i in range(len(fixed_pats_tensor)):
                p = fixed_pats_tensor[i]
                M[p[W], p[E]] += 1
            
            return self._chain_count(L, M, fixed_pats_tensor)
        else:
            # Fixed under reversal (di, True, flip)
            k = L // 2
            M_full = self._get_transfer_matrix(pats)
            
            if L % 2 == 0:
                v_init = torch.zeros(2, dtype=torch.double, device=self.device)
                for i in range(len(pats)): v_init[pats[i][E]] += 1
                
                if k > 1:
                    v_mid = v_init @ torch.matrix_power(M_full, k - 2)
                else:
                    v_mid = v_init if k == 1 else None
                
                count = 0.0
                for i in range(len(pats)):
                    p = pats[i]
                    # gp is the pattern g(p)
                    gp = p[perm]
                    if flip: gp = 1 - gp
                    # Hinge at center: p_{k-1}[E] == p_k[W]. 
                    # Since p_k = g(p_{k-1}), we need p_{k-1}[E] == g(p_{k-1})[W]
                    if p[E] == gp[W]:
                        if k == 1:
                            count += 1.0
                        else:
                            count += float(v_mid[p[W]])
                return count
            else:
                fixed_center = []
                for i in range(len(pats)):
                    p = pats[i]
                    gp = p[perm]
                    if flip: gp = 1 - gp
                    if torch.equal(p, gp): fixed_center.append(p)
                
                if not fixed_center: return 0.0
                if k == 0: return float(len(fixed_center))
                
                v_init = torch.zeros(2, dtype=torch.double, device=self.device)
                for i in range(len(pats)): v_init[pats[i][E]] += 1
                
                if k > 1:
                    v_mid = v_init @ torch.matrix_power(M_full, k - 2)
                else:
                    v_mid = v_init if k == 1 else None
                
                count = 0.0
                # p_{k-1} matches fixed_center p_k, and g(p_{k-1}) matches p_k
                for pc in fixed_center:
                    for i in range(len(pats)):
                        p_prev = pats[i]
                        if p_prev[E] == pc[W]:
                            gp_prev = p_prev[perm]
                            if flip: gp_prev = 1 - gp_prev
                            if gp_prev[W] == pc[E]:
                                if k == 1: count += 1.0
                                else: count += float(v_mid[p_prev[W]])
                return count

    def _chain_count(self, L, M, pats):
        if L == 1: return float(len(pats))
        v_init = torch.zeros(2, dtype=torch.double, device=self.device)
        for i in range(len(pats)): v_init[pats[i][E]] += 1
        v_end = torch.zeros(2, dtype=torch.double, device=self.device)
        for i in range(len(pats)): v_end[pats[i][W]] += 1
        if L > 2:
            res = v_init @ torch.matrix_power(M, L - 2)
            return float(torch.dot(res, v_end))
        else:
            # L=2 case
            count = 0.0
            for i in range(len(pats)):
                for j in range(len(pats)):
                    if pats[i][E] == pats[j][W]:
                        count += 1.0
            return count

    def solve(self, L):
        if L == 1:
             G0 = [(di, False) for di in range(8)]
        else:
             G0 = [(0, False), (4, False), (2, True), (5, True)]
        
        sum_V0 = sum(self.count_fixed(L, di, rev, flip=False) for di, rev in G0)
        sum_Both0 = sum(self.count_fixed(L, di, rev, flip=False, both_only=True) for di, rev in G0)
        sum_BothFlip = sum(self.count_fixed(L, di, rev, flip=True, both_only=True) for di, rev in G0)
        
        g0_size = len(G0)
        total_num = 2 * sum_V0 - sum_Both0 + sum_BothFlip
        return int(total_num // (2 * g0_size))


### Run Calculation (L=1..20)
We calculate the sequence. Notice how quickly it runs compared to the brute-force enumeration.

In [ ]:
solver = LinearSnakeSolver(device)
results = []
for L in range(1, 21):
    start = time.time()
    total = solver.solve(L)
    elapsed = time.time() - start
    results.append({"L": L, "Total": total, "Time(s)": elapsed})
    print(f"L={L:2d}: {total:20,d} ({elapsed:.4f}s)")

df = pd.DataFrame(results)
df